# Check the Sentence Embeddings

| Item | Description |
|---|---|
| Goal | Put the eight embedding shards from the T3.6.3 array back together into one matrix aligned row for row with the sentence table, and check it: every shard present and complete, unit-length rows, and a fresh re-embedding of a random sample that matches the stored vectors. |
| Input | `<scratch>/T3.6 - Firm Sentences/Interim/shards/manifest.csv`, `<scratch>/T3.6 - Firm Sentences/Interim/embeddings/shard_<k>.npy` and `shard_<k>.json` from T3.6.3, and `<scratch>/T3.6 - Firm Sentences/Processed/sentences.parquet` from T3.6.1. |
| Output | `<scratch>/T3.6 - Firm Sentences/Processed/sentence_embeddings.npy` (5,647,505 × 1024, float16, row $i$ is the embedding of row $i$ of `sentences.parquet`) and `output/T3.6.4/embedding_check.csv` (one row per shard with its run record and the recheck result). |
| Unit | A sentence. |
| Model | `BAAI/bge-large-en-v1.5`, CLS pooling, L2-normalized, as in every embedding step of this rebuild; the recheck runs on the GPU in float32 like the shards did. |
| Checks | Shard row counts match the manifest and sum to the sentence table; row ranges are contiguous; row norms are 1 within float16 precision; the cosine between each re-embedded sample sentence and its stored vector is at least 0.999. |

See `Topic Relevance Rebuild Plan.md` in the TEMP folder for the surrounding steps.


## Step 1. Import packages


In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from transformers import AutoModel, AutoTokenizer
from IPython.display import display


## Step 2. Define file locations and the check settings

The matrix stays on scratch beside the sentence table; the check record goes beside the notebook outputs.


In [2]:
PROJECT_FOLDER = Path("/project2/njia_1089/acroh/projects/LLM-Text-Analysis-and-Synthesis")
SCRATCH_FOLDER = Path("/scratch1/acroh/projects/LLM-Text-Analysis-and-Synthesis/T3-TEMP")
NOTEBOOK_SUBFOLDER = Path("T3 - Topic Relevance and Corporate Disclosure/TEMP")
NOTEBOOK_NAME = "T3.6.4"  # the identifier names this notebook's output folder
STAGE_FOLDER = SCRATCH_FOLDER / "T3.6 - Firm Sentences"
MANIFEST_FILE = STAGE_FOLDER / "Interim" / "shards" / "manifest.csv"
EMBEDDINGS_FOLDER = STAGE_FOLDER / "Interim" / "embeddings"
SENTENCES_FILE = STAGE_FOLDER / "Processed" / "sentences.parquet"
MATRIX_FILE = STAGE_FOLDER / "Processed" / "sentence_embeddings.npy"
MODEL_FOLDER = Path(
    "/scratch2/acroh/projects/llm_method/hf_cache/hub/models--BAAI--bge-large-en-v1.5"
    "/snapshots/d4aa6901d3a41ba39fb536a557fa166f842b0e09"
)
OUTPUT_FOLDER = PROJECT_FOLDER / NOTEBOOK_SUBFOLDER / "output" / NOTEBOOK_NAME

HIDDEN_SIZE = 1024  # the encoder's output width
MAX_TOKENS = 512  # the encoder's input limit
N_RECHECK = 200  # random sentences re-embedded and compared with the stored vectors
MIN_RECHECK_COSINE = 0.999  # float16 storage and batch padding allow this much drift
NORM_TOLERANCE = 1e-2  # unit length within float16 precision
RANDOM_SEED = 0
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)
print(f"Matrix: {MATRIX_FILE}")
print(f"Outputs: {OUTPUT_FOLDER}")


Matrix: /scratch1/acroh/projects/LLM-Text-Analysis-and-Synthesis/T3-TEMP/T3.6 - Firm Sentences/Processed/sentence_embeddings.npy
Outputs: /project2/njia_1089/acroh/projects/LLM-Text-Analysis-and-Synthesis/T3 - Topic Relevance and Corporate Disclosure/TEMP/output/T3.6.4


## Step 3. Load the manifest and the shard run records

Every shard listed in the manifest must have its embedding file and its run record, with the same row count.


In [3]:
manifest = pd.read_csv(MANIFEST_FILE).sort_values("shard_id").reset_index(drop=True)
records = []
for shard in manifest.itertuples(index=False):
    record_file = EMBEDDINGS_FOLDER / f"shard_{shard.shard_id}.json"
    embedding_file = EMBEDDINGS_FOLDER / f"shard_{shard.shard_id}.npy"
    assert record_file.exists() and embedding_file.exists(), f"shard {shard.shard_id} is missing"
    record = json.loads(record_file.read_text(encoding="utf-8"))
    assert record["n_rows"] == shard.n_rows and record["row_start"] == shard.row_start
    records.append(record)
records = pd.DataFrame(records)
assert (manifest["row_start"].iloc[1:].to_numpy() == manifest["row_end"].iloc[:-1].to_numpy() + 1).all()
n_rows_total = int(manifest["n_rows"].sum())
print(f"Shards: {len(manifest)}; rows: {n_rows_total:,}; GPUs used: {records['gpu'].value_counts().to_dict()}")
print(f"Minutes per shard: {(records['seconds'] / 60).round(1).tolist()}; sentences cut at {MAX_TOKENS} tokens: {int(records['n_truncated_at_max_tokens'].sum()):,}")
display(records[["shard_id", "n_rows", "gpu", "seconds", "n_truncated_at_max_tokens", "finished"]])


Shards: 8; rows: 5,647,505; GPUs used: {'NVIDIA A40': 8}
Minutes per shard: [28.1, 28.5, 27.8, 27.5, 26.9, 27.4, 26.8, 25.3]; sentences cut at 512 tokens: 515


,shard_id,n_rows,gpu,seconds,n_truncated_at_max_tokens,finished
0,0,705939,NVIDIA A40,1687.5,116,2026-10-25T13:49:19+00:00
1,1,705938,NVIDIA A40,1708.1,97,2026-10-25T13:51:54+00:00
2,2,705938,NVIDIA A40,1667.1,45,2026-10-25T14:18:09+00:00
3,3,705938,NVIDIA A40,1651.9,69,2026-10-25T14:17:53+00:00
4,4,705938,NVIDIA A40,1611.8,54,2026-10-25T14:19:53+00:00
5,5,705938,NVIDIA A40,1643.0,91,2026-10-25T14:23:31+00:00
6,6,705938,NVIDIA A40,1608.3,23,2026-10-25T14:22:48+00:00
7,7,705938,NVIDIA A40,1520.6,20,2026-10-25T14:28:30+00:00


## Step 4. Load the sentence table

The matrix must have exactly one row per sentence, in the table's order.


In [4]:
sentences = pd.read_parquet(SENTENCES_FILE)
assert len(sentences) == n_rows_total
assert not sentences.duplicated(["accessionNo", "sent_idx"]).any()
print(f"Sentences: {len(sentences):,} from {sentences['accessionNo'].nunique():,} filings")


Sentences: 5,647,505 from 18,779 filings


## Step 5. Assemble the matrix shard by shard

Write into one memory-mapped file so the 11.6 GB matrix never has to sit in memory twice. Each shard goes to its manifest row range; the norms of every row are checked as it is copied.


In [5]:
matrix = np.lib.format.open_memmap(MATRIX_FILE, mode="w+", dtype=np.float16, shape=(n_rows_total, HIDDEN_SIZE))
for shard in manifest.itertuples(index=False):
    shard_embeddings = np.load(EMBEDDINGS_FOLDER / f"shard_{shard.shard_id}.npy", mmap_mode="r")
    assert shard_embeddings.shape == (shard.n_rows, HIDDEN_SIZE) and shard_embeddings.dtype == np.float16
    norms = np.linalg.norm(shard_embeddings.astype(np.float32), axis=1)
    assert np.allclose(norms, 1, atol=NORM_TOLERANCE), f"shard {shard.shard_id} has rows off unit length"
    matrix[shard.row_start:shard.row_end + 1] = shard_embeddings
    print(f"  shard {shard.shard_id}: rows {shard.row_start:,}–{shard.row_end:,} copied; norms {norms.min():.4f}–{norms.max():.4f}")
matrix.flush()
del matrix
print(f"Saved: {MATRIX_FILE} ({MATRIX_FILE.stat().st_size / 1e9:.1f} GB)")


  shard 0: rows 0–705,938 copied; norms 0.9999–1.0001


  shard 1: rows 705,939–1,411,876 copied; norms 0.9999–1.0001


  shard 2: rows 1,411,877–2,117,814 copied; norms 0.9999–1.0001


  shard 3: rows 2,117,815–2,823,752 copied; norms 0.9999–1.0001


  shard 4: rows 2,823,753–3,529,690 copied; norms 0.9999–1.0001


  shard 5: rows 3,529,691–4,235,628 copied; norms 0.9999–1.0001


  shard 6: rows 4,235,629–4,941,566 copied; norms 0.9999–1.0001


  shard 7: rows 4,941,567–5,647,504 copied; norms 0.9999–1.0001


Saved: /scratch1/acroh/projects/LLM-Text-Analysis-and-Synthesis/T3-TEMP/T3.6 - Firm Sentences/Processed/sentence_embeddings.npy (11.6 GB)


## Step 6. Re-embed a random sample and compare

Load the encoder on the GPU in float32, embed 200 random sentences one batch at a time exactly as the shards did, and compare each with its stored vector. The cosine must be at least 0.999.


In [6]:
rng = np.random.default_rng(RANDOM_SEED)
sample_rows = np.sort(rng.choice(n_rows_total, size=N_RECHECK, replace=False))
matrix = np.load(MATRIX_FILE, mmap_mode="r")
stored = matrix[sample_rows].astype(np.float32)

tokenizer = AutoTokenizer.from_pretrained(MODEL_FOLDER, local_files_only=True)
assert torch.cuda.is_available(), "run this notebook on a GPU node"
device = torch.device("cuda")
model = AutoModel.from_pretrained(MODEL_FOLDER, local_files_only=True).to(device).eval()
with torch.no_grad():
    encoded = tokenizer(
        sentences["sentence"].iloc[sample_rows].tolist(), padding=True, truncation=True,
        max_length=MAX_TOKENS, return_tensors="pt",
    ).to(device)
    last_layer = model(**encoded).last_hidden_state
    fresh = torch.nn.functional.normalize(last_layer[:, 0], dim=-1).cpu().numpy()
recheck_cosine = (stored * fresh).sum(axis=1) / np.linalg.norm(stored, axis=1)
assert recheck_cosine.min() >= MIN_RECHECK_COSINE, f"lowest recheck cosine {recheck_cosine.min():.5f}"
print(f"Device: {torch.cuda.get_device_name(device)}; rechecked {N_RECHECK} sentences; cosine with stored vectors: min {recheck_cosine.min():.5f}, median {np.median(recheck_cosine):.5f}")


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Device: NVIDIA A40; rechecked 200 sentences; cosine with stored vectors: min 1.00000, median 1.00000


## Step 7. Save the check record

One row per shard with its run record, plus the recheck result repeated on every row for the file to stand alone.


In [7]:
check = records[["shard_id", "n_rows", "row_start", "row_end", "gpu", "seconds", "n_truncated_at_max_tokens", "finished"]].copy()
check["recheck_sentences"] = N_RECHECK
check["recheck_min_cosine"] = float(recheck_cosine.min())
check["matrix_file"] = str(MATRIX_FILE)
check_file = OUTPUT_FOLDER / "embedding_check.csv"
check.to_csv(check_file, index=False)
print(f"Saved: {check_file}")
display(check)


Saved: /project2/njia_1089/acroh/projects/LLM-Text-Analysis-and-Synthesis/T3 - Topic Relevance and Corporate Disclosure/TEMP/output/T3.6.4/embedding_check.csv


,shard_id,n_rows,row_start,row_end,gpu,seconds,n_truncated_at_max_tokens,finished,recheck_sentences,recheck_min_cosine,matrix_file
0,0,705939,0,705938,NVIDIA A40,1687.5,116,2026-10-25T13:49:19+00:00,200,1.0,/scratch1/acroh/projects/LLM-Text-Analysis-and...
1,1,705938,705939,1411876,NVIDIA A40,1708.1,97,2026-10-25T13:51:54+00:00,200,1.0,/scratch1/acroh/projects/LLM-Text-Analysis-and...
2,2,705938,1411877,2117814,NVIDIA A40,1667.1,45,2026-10-25T14:18:09+00:00,200,1.0,/scratch1/acroh/projects/LLM-Text-Analysis-and...
3,3,705938,2117815,2823752,NVIDIA A40,1651.9,69,2026-10-25T14:17:53+00:00,200,1.0,/scratch1/acroh/projects/LLM-Text-Analysis-and...
4,4,705938,2823753,3529690,NVIDIA A40,1611.8,54,2026-10-25T14:19:53+00:00,200,1.0,/scratch1/acroh/projects/LLM-Text-Analysis-and...
5,5,705938,3529691,4235628,NVIDIA A40,1643.0,91,2026-10-25T14:23:31+00:00,200,1.0,/scratch1/acroh/projects/LLM-Text-Analysis-and...
6,6,705938,4235629,4941566,NVIDIA A40,1608.3,23,2026-10-25T14:22:48+00:00,200,1.0,/scratch1/acroh/projects/LLM-Text-Analysis-and...
7,7,705938,4941567,5647504,NVIDIA A40,1520.6,20,2026-10-25T14:28:30+00:00,200,1.0,/scratch1/acroh/projects/LLM-Text-Analysis-and...
